In [1]:
import sqlite3
import random
from datetime import datetime, timedelta

conn = sqlite3.connect("ecommerce.db")
cursor = conn.cursor()

cursor.executescript("""
DROP TABLE IF EXISTS customers;
DROP TABLE IF EXISTS products;
DROP TABLE IF EXISTS orders;
DROP TABLE IF EXISTS order_items;
DROP TABLE IF EXISTS returns;
DROP TABLE IF EXISTS reviews;

CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    name TEXT,
    city TEXT,
    signup_date TEXT,
    segment TEXT
);

CREATE TABLE products (
    product_id INTEGER PRIMARY KEY,
    name TEXT,
    category TEXT,
    price REAL
);

CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    order_date TEXT,
    status TEXT,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
);

CREATE TABLE order_items (
    order_item_id INTEGER PRIMARY KEY,
    order_id INTEGER,
    product_id INTEGER,
    quantity INTEGER,
    unit_price REAL,
    FOREIGN KEY (order_id) REFERENCES orders(order_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id)
);

CREATE TABLE returns (
    return_id INTEGER PRIMARY KEY,
    order_item_id INTEGER,
    return_date TEXT,
    reason TEXT,
    FOREIGN KEY (order_item_id) REFERENCES order_items(order_item_id)
);

CREATE TABLE reviews (
    review_id INTEGER PRIMARY KEY,
    product_id INTEGER,
    customer_id INTEGER,
    rating INTEGER,
    review_date TEXT,
    FOREIGN KEY (product_id) REFERENCES products(product_id),
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
);
""")

conn.commit()
print("Schema created")

Schema created


In [3]:
import random
from datetime import datetime, timedelta

random.seed(42)  # reproducible dataset

# --- Customers ---
cities = ["Mumbai", "Delhi", "Bangalore", "Chennai", "Kolkata", "Hyderabad", "Pune"]
segments = ["Retail", "Wholesale", "VIP"]
first_names = ["Aarav","Vihaan","Aditi","Ishaan","Diya","Kabir","Ananya","Arjun","Meera","Rohan",
               "Priya","Karan","Neha","Rahul","Sanya","Aryan","Pooja","Vikram","Tanvi","Dev"]
last_names = ["Sharma","Verma","Iyer","Reddy","Nair","Gupta","Joshi","Mehta","Rao","Singh"]

n_customers = 150
customers = []
start = datetime(2024, 1, 1)
for cid in range(1, n_customers + 1):
    name = f"{random.choice(first_names)} {random.choice(last_names)}"
    city = random.choice(cities)
    signup = start + timedelta(days=random.randint(0, 600))
    segment = random.choices(segments, weights=[70, 20, 10])[0]
    customers.append((cid, name, city, signup.strftime("%Y-%m-%d"), segment))

cursor.executemany("INSERT INTO customers VALUES (?,?,?,?,?)", customers)

# --- Products ---
categories = ["Electronics", "Home", "Apparel", "Books", "Sports"]
product_names = {
    "Electronics": ["Wireless Earbuds","Bluetooth Speaker","Power Bank","Smart Watch","USB Cable"],
    "Home": ["Table Lamp","Cushion Set","Storage Box","Wall Clock","Kitchen Scale"],
    "Apparel": ["Cotton T-Shirt","Denim Jacket","Running Shoes","Wool Sweater","Sports Cap"],
    "Books": ["Fiction Novel","Business Guide","Cookbook","Biography","Self-Help Book"],
    "Sports": ["Yoga Mat","Dumbbell Set","Cricket Bat","Football","Resistance Bands"]
}
products = []
pid = 1
for cat, names in product_names.items():
    for n in names:
        price = round(random.uniform(200, 5000), 2)
        products.append((pid, n, cat, price))
        pid += 1
# duplicate a few products to widen the catalog
for _ in range(15):
    base = random.choice(products)
    price = round(base[3] * random.uniform(0.8, 1.2), 2)
    products.append((pid, base[1] + " (v2)", base[2], price))
    pid += 1

cursor.executemany("INSERT INTO products VALUES (?,?,?,?)", products)

n_products = len(products)

# --- Orders + Order Items ---
# deliberately skew: a small group of "repeat" customers order far more often (trap for average-based questions)
repeat_customers = random.sample(range(1, n_customers + 1), 20)

order_id = 1
order_item_id = 1
orders = []
order_items = []
statuses = ["Completed", "Completed", "Completed", "Cancelled", "Pending"]

for cid in range(1, n_customers + 1):
    n_orders = random.randint(3, 10) if cid in repeat_customers else random.randint(0, 3)
    for _ in range(n_orders):
        odate = start + timedelta(days=random.randint(0, 700))
        status = random.choices(statuses, weights=[60, 15, 15, 5, 5])[0]
        orders.append((order_id, cid, odate.strftime("%Y-%m-%d"), status))

        n_items = random.randint(1, 4)  # multi-item orders — the join trap
        chosen_products = random.sample(range(1, n_products + 1), n_items)
        for prod_id in chosen_products:
            qty = random.randint(1, 3)
            unit_price = next(p[3] for p in products if p[0] == prod_id)
            order_items.append((order_item_id, order_id, prod_id, qty, unit_price))
            order_item_id += 1
        order_id += 1

cursor.executemany("INSERT INTO orders VALUES (?,?,?,?)", orders)
cursor.executemany("INSERT INTO order_items VALUES (?,?,?,?,?)", order_items)

# --- Returns ---
# cluster returns around a couple of specific products (trap: "why are returns high" question)
problem_products = random.sample(range(1, n_products + 1), 3)
reasons = ["Defective", "Wrong item", "Changed mind", "Late delivery", "Quality issue"]

returns = []
return_id = 1
for item in order_items:
    oi_id, o_id, p_id, qty, price = item
    base_chance = 0.15 if p_id in problem_products else 0.03
    if random.random() < base_chance:
        order_date = next(o[2] for o in orders if o[0] == o_id)
        rdate = datetime.strptime(order_date, "%Y-%m-%d") + timedelta(days=random.randint(2, 20))
        reason = random.choice(reasons) if p_id not in problem_products else "Defective"
        returns.append((return_id, oi_id, rdate.strftime("%Y-%m-%d"), reason))
        return_id += 1

cursor.executemany("INSERT INTO returns VALUES (?,?,?,?)", returns)

# --- Reviews ---
reviews = []
review_id = 1
for cid in range(1, n_customers + 1):
    n_reviews = random.randint(0, 5)
    reviewed_products = random.sample(range(1, n_products + 1), min(n_reviews, n_products))
    for p_id in reviewed_products:
        rating = random.choices([1,2,3,4,5], weights=[5,5,15,35,40])[0]
        if p_id in problem_products:
            rating = random.choices([1,2,3,4,5], weights=[35,25,20,15,5])[0]  # problem products skew low
        rdate = start + timedelta(days=random.randint(0, 700))
        reviews.append((review_id, p_id, cid, rating, rdate.strftime("%Y-%m-%d")))
        review_id += 1

cursor.executemany("INSERT INTO reviews VALUES (?,?,?,?,?)", reviews)

conn.commit()

print(f"Customers: {len(customers)}")
print(f"Products: {len(products)}")
print(f"Orders: {len(orders)}")
print(f"Order items: {len(order_items)}")
print(f"Returns: {len(returns)}")
print(f"Reviews: {len(reviews)}")
print(f"Repeat customers (trap for avg-based questions): {sorted(repeat_customers)[:5]}...")
print(f"Problem products (trap for return-rate questions): {problem_products}")

Customers: 150
Products: 40
Orders: 371
Order items: 924
Returns: 37
Reviews: 378
Repeat customers (trap for avg-based questions): [14, 17, 19, 22, 30]...
Problem products (trap for return-rate questions): [33, 2, 1]


In [4]:
import pandas as pd

# Sanity check 1: total revenue (the join trap in action)
# WRONG way — this is what a naive AI-generated query often does
wrong = pd.read_sql_query("""
    SELECT SUM(oi.quantity * oi.unit_price) as revenue
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
""", conn)
print("Naive total revenue:", wrong['revenue'][0])

# CORRECT way — same query, but this one's actually right since order_items
# is the correct grain (no duplication happens here, but this becomes critical
# once you add a JOIN to another one-to-many table like returns)
correct = pd.read_sql_query("""
    SELECT SUM(oi.quantity * oi.unit_price) as revenue
    FROM order_items oi
""", conn)
print("Correct total revenue:", correct['revenue'][0])

# Sanity check 2: the real join trap — revenue per order when you also join returns
# This WILL double count, because an order with 2 returned items joins twice
trap = pd.read_sql_query("""
    SELECT SUM(oi.quantity * oi.unit_price) as revenue
    FROM order_items oi
    LEFT JOIN returns r ON oi.order_item_id = r.order_item_id
""", conn)
print("Trap query revenue (should be inflated if any returns exist):", trap['revenue'][0])

# Sanity check 3: confirm problem products really do have higher return rates
problem_check = pd.read_sql_query("""
    SELECT p.product_id, p.name,
           COUNT(DISTINCT oi.order_item_id) as times_ordered,
           COUNT(DISTINCT r.return_id) as times_returned,
           ROUND(100.0 * COUNT(DISTINCT r.return_id) / COUNT(DISTINCT oi.order_item_id), 1) as return_rate_pct
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    LEFT JOIN returns r ON oi.order_item_id = r.order_item_id
    GROUP BY p.product_id
    ORDER BY return_rate_pct DESC
    LIMIT 10
""", conn)
print(problem_check)

Naive total revenue: 4340603.87
Correct total revenue: 4340603.87
Trap query revenue (should be inflated if any returns exist): 4340603.87
   product_id               name  times_ordered  times_returned  \
0          33     Biography (v2)             18               5   
1           9         Wall Clock             21               4   
2           2  Bluetooth Speaker             18               2   
3           1   Wireless Earbuds             19               2   
4          10      Kitchen Scale             29               3   
5           5          USB Cable             31               3   
6          36   Smart Watch (v2)             21               2   
7           3         Power Bank             22               2   
8           7        Cushion Set             16               1   
9          15         Sports Cap             19               1   

   return_rate_pct  
0             27.8  
1             19.0  
2             11.1  
3             10.5  
4             10.3

In [5]:
# The REAL trap: average order value
# WRONG — averaging at the item level, not the order level
wrong_avg = pd.read_sql_query("""
    SELECT AVG(oi.quantity * oi.unit_price) as avg_order_value
    FROM order_items oi
""", conn)
print("Wrong avg order value (item-level average):", round(wrong_avg['avg_order_value'][0], 2))

# CORRECT — sum per order first, then average across orders
correct_avg = pd.read_sql_query("""
    SELECT AVG(order_total) as avg_order_value
    FROM (
        SELECT order_id, SUM(quantity * unit_price) as order_total
        FROM order_items
        GROUP BY order_id
    )
""", conn)
print("Correct avg order value (order-level average):", round(correct_avg['avg_order_value'][0], 2))

# a related real trap: counting orders after joining to order_items inflates the count,
# since multi-item orders appear once per item
wrong_count = pd.read_sql_query("""
    SELECT COUNT(*) as order_count
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
""", conn)
print("Wrong order count (inflated by items):", wrong_count['order_count'][0])

correct_count = pd.read_sql_query("SELECT COUNT(*) as order_count FROM orders", conn)
print("Correct order count:", correct_count['order_count'][0])

Wrong avg order value (item-level average): 4697.62
Correct avg order value (order-level average): 11699.74
Wrong order count (inflated by items): 924
Correct order count: 371


In [4]:
import pandas as pd
print("pandas ready")

pandas ready


In [6]:
import sqlite3
conn = sqlite3.connect("ecommerce.db")
print("Connected to database")

Connected to database


In [7]:
schema_description = """
Tables:

customers (customer_id, name, city, signup_date, segment)
products (product_id, name, category, price)
orders (order_id, customer_id, order_date, status)
order_items (order_item_id, order_id, product_id, quantity, unit_price)
returns (return_id, order_item_id, return_date, reason)
reviews (review_id, product_id, customer_id, rating, review_date)

Relationships:
- orders.customer_id -> customers.customer_id
- order_items.order_id -> orders.order_id
- order_items.product_id -> products.product_id
- returns.order_item_id -> order_items.order_item_id
- reviews.product_id -> products.product_id
- reviews.customer_id -> customers.customer_id

Important: order_items is the line-item level. An order can have multiple 
order_items. To compute order-level metrics (like average order value), you 
must aggregate to the order level BEFORE averaging across orders — do not 
average across order_items directly, since that mixes item-level and 
order-level granularity.
"""

def ask_sql_agent(question):
    prompt = f"""You are a SQL analyst. Given this database schema:

{schema_description}

Write a single SQLite query that answers this question: "{question}"

Return ONLY the SQL query, no explanation, no markdown formatting, no 
backticks. Just the raw SQL.
"""
    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": prompt}]
    )
    sql = response.choices[0].message.content.strip()
    # strip markdown code fences if the model adds them anyway
    sql = sql.replace("```sql", "").replace("```", "").strip()
    return sql

# Test 1: the exact trap question
question = "What is the average order value?"
sql = ask_sql_agent(question)
print("Generated SQL:")
print(sql)
print()

result = pd.read_sql_query(sql, conn)
print("Result:")
print(result)

Generated SQL:
SELECT AVG(order_total) AS average_order_value
FROM (
    SELECT order_id, SUM(quantity * unit_price) AS order_total
    FROM order_items
    GROUP BY order_id
);

Result:
   average_order_value
0         11699.740889


In [2]:
from groq import Groq
client = Groq()
print("Groq client ready")

Groq client ready


In [8]:
schema_description_no_hint = """
Tables:

customers (customer_id, name, city, signup_date, segment)
products (product_id, name, category, price)
orders (order_id, customer_id, order_date, status)
order_items (order_item_id, order_id, product_id, quantity, unit_price)
returns (return_id, order_item_id, return_date, reason)
reviews (review_id, product_id, customer_id, rating, review_date)

Relationships:
- orders.customer_id -> customers.customer_id
- order_items.order_id -> orders.order_id
- order_items.product_id -> products.product_id
- returns.order_item_id -> order_items.order_item_id
- reviews.product_id -> products.product_id
- reviews.customer_id -> customers.customer_id
"""

def ask_sql_agent_no_hint(question):
    prompt = f"""You are a SQL analyst. Given this database schema:

{schema_description_no_hint}

Write a single SQLite query that answers this question: "{question}"

Return ONLY the SQL query, no explanation, no markdown formatting, no 
backticks. Just the raw SQL.
"""
    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": prompt}]
    )
    sql = response.choices[0].message.content.strip()
    sql = sql.replace("```sql", "").replace("```", "").strip()
    return sql

question = "What is the average order value?"
sql_no_hint = ask_sql_agent_no_hint(question)
print("Generated SQL (no granularity hint):")
print(sql_no_hint)
print()

result_no_hint = pd.read_sql_query(sql_no_hint, conn)
print("Result:")
print(result_no_hint)
print()
print("Compare to known-correct value: 11699.74")

Generated SQL (no granularity hint):
SELECT AVG(order_total) AS average_order_value
FROM (
    SELECT o.order_id, SUM(oi.quantity * oi.unit_price) AS order_total
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    GROUP BY o.order_id
) AS order_totals;

Result:
   average_order_value
0         11699.740889

Compare to known-correct value: 11699.74


In [9]:
test_questions = [
    "Which customers have never returned anything?",
    "What percentage of orders included more than one product?",
    "Which product category has the highest return rate?",
    "How much revenue came from customers who signed up in their first order month?"
]

results = {}

for q in test_questions:
    print("=" * 80)
    print("QUESTION:", q)
    print("=" * 80)
    sql = ask_sql_agent_no_hint(q)
    print("Generated SQL:")
    print(sql)
    print()
    try:
        result = pd.read_sql_query(sql, conn)
        print("Result:")
        print(result.head(10))
        results[q] = {"sql": sql, "result": result, "error": None}
    except Exception as e:
        print("ERROR:", e)
        results[q] = {"sql": sql, "result": None, "error": str(e)}
    print()

QUESTION: Which customers have never returned anything?
Generated SQL:
SELECT c.customer_id, c.name
FROM customers c
WHERE NOT EXISTS (
    SELECT 1
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    JOIN returns r ON r.order_item_id = oi.order_item_id
    WHERE o.customer_id = c.customer_id
);

Result:
   customer_id           name
0            1  Ishaan Sharma
1            2     Diya Verma
2            3   Rahul Sharma
3            5   Vikram Joshi
4            6     Aarav Iyer
5            7     Diya Reddy
6            8     Neha Verma
7           10     Neha Verma
8           11      Dev Gupta
9           12   Vihaan Reddy

QUESTION: What percentage of orders included more than one product?
Generated SQL:
SELECT 
    100.0 * SUM(CASE WHEN prod_cnt > 1 THEN 1 ELSE 0 END) / COUNT(*) AS percentage_multi_product_orders
FROM (
    SELECT o.order_id, COUNT(DISTINCT oi.product_id) AS prod_cnt
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    

In [10]:
# Verify Q2 independently with pandas
items_df = pd.read_sql_query("SELECT * FROM order_items", conn)
products_per_order = items_df.groupby("order_id")["product_id"].nunique()
multi_product_pct = (products_per_order > 1).mean() * 100
print("Independent check — % multi-product orders:", round(multi_product_pct, 2))
print("Agent's answer was: 74.39")

# Verify Q3 independently with pandas
oi_df = pd.read_sql_query("SELECT oi.order_item_id, oi.product_id, p.category FROM order_items oi JOIN products p ON oi.product_id = p.product_id", conn)
ret_df = pd.read_sql_query("SELECT order_item_id FROM returns", conn)
oi_df["returned"] = oi_df["order_item_id"].isin(ret_df["order_item_id"])
category_return_rates = oi_df.groupby("category")["returned"].mean().sort_values(ascending=False)
print("\nIndependent check — return rate by category:")
print(category_return_rates)
print("\nAgent said Electronics was highest at 0.063")

Independent check — % multi-product orders: 74.39
Agent's answer was: 74.39

Independent check — return rate by category:
category
Electronics    0.063107
Home           0.055249
Books          0.036437
Sports         0.023121
Apparel        0.008547
Name: returned, dtype: float64

Agent said Electronics was highest at 0.063


In [11]:
hard_questions = [
    "For each product category, rank products by total revenue within that category, and show the top 2 products per category.",
    "What is the running total of revenue by month for 2025?",
    "Which customers have a personal return rate higher than the average return rate across all customers?"
]

hard_results = {}

for q in hard_questions:
    print("=" * 80)
    print("QUESTION:", q)
    print("=" * 80)
    sql = ask_sql_agent_no_hint(q)
    print("Generated SQL:")
    print(sql)
    print()
    try:
        result = pd.read_sql_query(sql, conn)
        print("Result:")
        print(result.head(15))
        hard_results[q] = {"sql": sql, "result": result, "error": None}
    except Exception as e:
        print("ERROR:", e)
        hard_results[q] = {"sql": sql, "result": None, "error": str(e)}
    print()

QUESTION: For each product category, rank products by total revenue within that category, and show the top 2 products per category.
Generated SQL:
SELECT
    category,
    product_id,
    name,
    total_revenue,
    rank
FROM (
    SELECT
        p.category,
        p.product_id,
        p.name,
        SUM(oi.quantity * oi.unit_price) AS total_revenue,
        ROW_NUMBER() OVER (PARTITION BY p.category ORDER BY SUM(oi.quantity * oi.unit_price) DESC) AS rank
    FROM products p
    JOIN order_items oi ON p.product_id = oi.product_id
    GROUP BY p.category, p.product_id, p.name
) AS ranked
WHERE rank <= 2
ORDER BY category, rank;

Result:
      category  product_id               name  total_revenue  rank
0      Apparel          34  Denim Jacket (v2)      108447.90     1
1      Apparel          14       Wool Sweater      104997.31     2
2        Books          18           Cookbook      304013.63     1
3        Books          16      Fiction Novel      234246.94     2
4  Electronics   

In [12]:
# Verify Q1 — top 2 products by revenue per category
items_full = pd.read_sql_query("""
    SELECT p.category, p.product_id, p.name, oi.quantity * oi.unit_price AS line_revenue
    FROM order_items oi JOIN products p ON oi.product_id = p.product_id
""", conn)
rev_by_product = items_full.groupby(["category","product_id","name"])["line_revenue"].sum().reset_index()
top2 = rev_by_product.sort_values(["category","line_revenue"], ascending=[True,False]).groupby("category").head(2)
print(top2)

# Verify Q2 — running total by month, 2025
orders_2025 = pd.read_sql_query("""
    SELECT o.order_date, oi.quantity * oi.unit_price AS line_revenue
    FROM orders o JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.order_date >= '2025-01-01' AND o.order_date < '2026-01-01'
""", conn)
orders_2025["month"] = orders_2025["order_date"].str[:7]
monthly = orders_2025.groupby("month")["line_revenue"].sum().sort_index()
print(monthly.cumsum())

       category  product_id               name  line_revenue
5       Apparel          34  Denim Jacket (v2)     108447.90
3       Apparel          14       Wool Sweater     104997.31
8         Books          18           Cookbook     304013.63
6         Books          16      Fiction Novel     234246.94
20  Electronics           5          USB Cable     189080.10
21  Electronics          32   Smart Watch (v2)     136473.12
27         Home           8        Storage Box     137439.92
32         Home          37    Wall Clock (v2)     109160.04
38       Sports          26      Football (v2)     252599.60
36       Sports          24           Football     168459.36
month
2025-01     314853.01
2025-02     392509.32
2025-03     573266.06
2025-04     758920.48
2025-05     895026.53
2025-06    1031028.17
2025-07    1246350.02
2025-08    1431370.45
2025-09    1517893.23
2025-10    1700318.09
2025-11    1983649.65
Name: line_revenue, dtype: float64


In [13]:
# Two competing definitions of "average return rate"
customer_orders = pd.read_sql_query("""
    SELECT c.customer_id, c.name, oi.order_item_id, r.return_id
    FROM customers c
    JOIN orders o ON o.customer_id = c.customer_id
    JOIN order_items oi ON oi.order_id = o.order_id
    LEFT JOIN returns r ON r.order_item_id = oi.order_item_id
""", conn)

per_customer = customer_orders.groupby(["customer_id","name"]).agg(
    total_items=("order_item_id","count"),
    total_returns=("return_id","count")
).reset_index()
per_customer["return_rate"] = per_customer["total_returns"] / per_customer["total_items"]

unweighted_avg = per_customer["return_rate"].mean()
weighted_avg = per_customer["total_returns"].sum() / per_customer["total_items"].sum()

print("Unweighted average (average of each customer's own rate):", round(unweighted_avg, 4))
print("Weighted average (total returns / total items, everyone pooled):", round(weighted_avg, 4))

above_unweighted = set(per_customer[per_customer["return_rate"] > unweighted_avg]["customer_id"])
above_weighted = set(per_customer[per_customer["return_rate"] > weighted_avg]["customer_id"])

print(f"\nCustomers above unweighted avg: {len(above_unweighted)}")
print(f"Customers above weighted avg: {len(above_weighted)}")
print(f"Agent's answer had 15 customers, using unweighted avg")
print(f"Customers in one list but not the other: {above_unweighted.symmetric_difference(above_weighted)}")

Unweighted average (average of each customer's own rate): 0.0459
Weighted average (total returns / total items, everyone pooled): 0.04

Customers above unweighted avg: 29
Customers above weighted avg: 31
Agent's answer had 15 customers, using unweighted avg
Customers in one list but not the other: {78, 39}


In [14]:
# Pull the SQL's own intermediate CTE values directly, so we can compare
# apples to apples against the pandas per_customer table
sql_debug = """
WITH customer_orders AS (
    SELECT c.customer_id,
           COUNT(oi.order_item_id) AS total_items,
           SUM(CASE WHEN r.return_id IS NOT NULL THEN 1 ELSE 0 END) AS total_returns
    FROM customers c
    JOIN orders o ON o.customer_id = c.customer_id
    JOIN order_items oi ON oi.order_id = o.order_id
    LEFT JOIN returns r ON r.order_item_id = oi.order_item_id
    GROUP BY c.customer_id
),
customer_rates AS (
    SELECT customer_id,
           total_items,
           total_returns,
           CAST(total_returns AS REAL) / total_items AS return_rate
    FROM customer_orders
)
SELECT * FROM customer_rates ORDER BY customer_id
"""
sql_rates = pd.read_sql_query(sql_debug, conn)
print("Number of customers in SQL's own CTE:", len(sql_rates))
print("Number of customers in pandas per_customer:", len(per_customer))

# merge and compare directly, customer by customer
merged = sql_rates.merge(per_customer, on="customer_id", suffixes=("_sql", "_pandas"))
print("\nRows where total_items differs between SQL and pandas:")
print(merged[merged["total_items_sql"] != merged["total_items"]])

print("\nRows where total_returns differs between SQL and pandas:")
print(merged[merged["total_returns_sql"] != merged["total_returns"]])

print("\nSQL's own computed average return rate:", sql_rates["return_rate"].mean())
print("Pandas unweighted average (from before):", round(unweighted_avg, 4))

Number of customers in SQL's own CTE: 129
Number of customers in pandas per_customer: 129

Rows where total_items differs between SQL and pandas:


KeyError: 'total_items'

In [15]:
# 1. What does the SQL's own CTE actually compute as the average?
print("SQL's own average return rate:", sql_rates["return_rate"].mean())
print("Number of customers above that average, computed directly in SQL:")
above_in_sql = sql_rates[sql_rates["return_rate"] > sql_rates["return_rate"].mean()]
print(len(above_in_sql))

# 2. Re-run the EXACT original agent query fresh, right now, to see if it's reproducible
original_query = """
WITH customer_orders AS (
    SELECT c.customer_id,
           COUNT(oi.order_item_id) AS total_items,
           SUM(CASE WHEN r.return_id IS NOT NULL THEN 1 ELSE 0 END) AS total_returns
    FROM customers c
    JOIN orders o ON o.customer_id = c.customer_id
    JOIN order_items oi ON oi.order_id = o.order_id
    LEFT JOIN returns r ON r.order_item_id = oi.order_item_id
    GROUP BY c.customer_id
),
customer_rates AS (
    SELECT customer_id,
           CASE WHEN total_items = 0 THEN 0.0
                ELSE CAST(total_returns AS REAL) / total_items
           END AS return_rate
    FROM customer_orders
),
avg_rate AS (
    SELECT AVG(return_rate) AS avg_return_rate FROM customer_rates
)
SELECT c.customer_id, c.name
FROM customer_rates cr
JOIN avg_rate ar ON cr.return_rate > ar.avg_return_rate
JOIN customers c ON c.customer_id = cr.customer_id;
"""
rerun = pd.read_sql_query(original_query, conn)
print("\nRe-running the exact original query, right now:", len(rerun), "customers")
print(rerun)

SQL's own average return rate: 0.045940685778121715
Number of customers above that average, computed directly in SQL:
29

Re-running the exact original query, right now: 29 customers
    customer_id          name
0             4   Pooja Singh
1             9  Vihaan Mehta
2            15   Meera Verma
3            22    Vikram Rao
4            26   Rahul Singh
5            29   Priya Verma
6            30    Aarav Nair
7            35   Arjun Singh
8            40   Pooja Mehta
9            43  Vihaan Gupta
10           46   Rahul Reddy
11           47   Karan Joshi
12           48  Vihaan Verma
13           58  Priya Sharma
14           63   Tanvi Singh
15           74  Vihaan Verma
16           81     Karan Rao
17           88   Meera Verma
18           96   Priya Singh
19           97     Rohan Rao
20          101   Tanvi Singh
21          107   Tanvi Reddy
22          113   Pooja Joshi
23          117  Vikram Verma
24          118    Diya Reddy
25          128    Rahul Iyer
26     

In [17]:
def ask_sql_agent_with_self_check(question, conn):
    # Step 1: generate SQL
    sql = ask_sql_agent_no_hint(question)
    
    # Step 2: execute it
    try:
        result = pd.read_sql_query(sql, conn)
    except Exception as e:
        result = None
        error = str(e)
    else:
        error = None
    
    # Step 3: ask the model to review its own work
    review_prompt = f"""You wrote this SQL to answer the question: "{question}"

SQL:
{sql}

{"It failed with this error: " + error if error else "It ran and returned this result: " + result.to_string()}

Review your own query critically. Check specifically for:
- Aggregating at the wrong level of granularity (e.g. averaging line items instead of order totals)
- Missing GROUP BY that causes duplicate counting after a JOIN
- Using INNER JOIN when LEFT JOIN was needed (e.g. for "customers who never..." questions)
- Whether the result plausibly answers the question asked

Respond with ONLY one of:
"OK" if the query and result are correct
"RETRY: <specific reason>" if something is wrong and needs fixing
"""
    review = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": review_prompt}]
    ).choices[0].message.content.strip()
    
    retried = False
    if review.startswith("RETRY"):
        retried = True
        fix_prompt = f"""Your previous SQL for the question "{question}" had this issue: {review}

Previous SQL:
{sql}

Write a corrected SQLite query. Return ONLY the SQL, no explanation, no markdown.
"""
        sql = client.chat.completions.create(
            model="openai/gpt-oss-120b",
            messages=[{"role": "user", "content": fix_prompt}]
        ).choices[0].message.content.strip().replace("```sql", "").replace("```", "").strip()
        try:
            result = pd.read_sql_query(sql, conn)
            error = None
        except Exception as e:
            result = None
            error = str(e)
    
    return {
        "question": question,
        "sql": sql,
        "result": result,
        "error": error,
        "review": review,
        "retried": retried
    }

# test it
outcome = ask_sql_agent_with_self_check("What is the average order value?", conn)
print("Retried:", outcome["retried"])
print("Review:", outcome["review"])
print("Final SQL:", outcome["sql"])
print("Result:")
print(outcome["result"])

Retried: False
Review: OK
Final SQL: SELECT AVG(order_total) AS average_order_value
FROM (
    SELECT o.order_id, SUM(oi.quantity * oi.unit_price) AS order_total
    FROM orders o
    JOIN order_items oi ON oi.order_id = o.order_id
    GROUP BY o.order_id
) AS order_totals;
Result:
   average_order_value
0         11699.740889


In [18]:
outcome2 = ask_sql_agent_with_self_check(
    "What is the total revenue per customer segment, and how many customers are in each segment?",
    conn
)
print("Retried:", outcome2["retried"])
print("Review:", outcome2["review"])
print("Final SQL:", outcome2["sql"])
print("Result:")
print(outcome2["result"])


Retried: False
Review: OK
Final SQL: SELECT c.segment,
       COALESCE(SUM(oi.quantity * oi.unit_price), 0) AS total_revenue,
       COUNT(DISTINCT c.customer_id) AS customer_count
FROM customers c
LEFT JOIN orders o ON o.customer_id = c.customer_id
LEFT JOIN order_items oi ON oi.order_id = o.order_id
GROUP BY c.segment;
Result:
     segment  total_revenue  customer_count
0     Retail     3201699.32             103
1        VIP      331059.26              12
2  Wholesale      807845.29              35


In [19]:
outcome2 = ask_sql_agent_with_self_check(
    "What is the total revenue per customer segment, and how many customers are in each segment?",
    conn
)
print("Retried:", outcome2["retried"])
print("Review:", outcome2["review"])
print("Final SQL:", outcome2["sql"])
print("Result:")
print(outcome2["result"])

Retried: False
Review: OK
Final SQL: SELECT 
    c.segment,
    COUNT(DISTINCT c.customer_id) AS customer_count,
    COALESCE(SUM(oi.quantity * oi.unit_price), 0) AS total_revenue
FROM customers c
LEFT JOIN orders o ON o.customer_id = c.customer_id
LEFT JOIN order_items oi ON oi.order_id = o.order_id
GROUP BY c.segment;
Result:
     segment  customer_count  total_revenue
0     Retail             103     3201699.32
1        VIP              12      331059.26
2  Wholesale              35      807845.29


In [20]:
def dual_verify(question, sql, sql_result, conn):
    """
    Ask the model to write an INDEPENDENT pandas computation for the same
    question, without showing it the SQL it already wrote, so the two paths
    don't share the same mistake.
    """
    pandas_prompt = f"""Given these pandas DataFrames loaded from a database:

customers_df, products_df, orders_df, order_items_df, returns_df, reviews_df

Each has the same columns as a SQL table with that name (minus the _df suffix).

Write pandas code (not SQL) that answers this question: "{question}"

The code must end by assigning the final answer to a variable called `pandas_answer`.
Return ONLY the Python code, no explanation, no markdown, no backticks.
"""
    pandas_code = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": pandas_prompt}]
    ).choices[0].message.content.strip().replace("```python", "").replace("```", "").strip()

    # load fresh dataframes so the pandas path is truly independent
    customers_df = pd.read_sql_query("SELECT * FROM customers", conn)
    products_df = pd.read_sql_query("SELECT * FROM products", conn)
    orders_df = pd.read_sql_query("SELECT * FROM orders", conn)
    order_items_df = pd.read_sql_query("SELECT * FROM order_items", conn)
    returns_df = pd.read_sql_query("SELECT * FROM returns", conn)
    reviews_df = pd.read_sql_query("SELECT * FROM reviews", conn)

    local_scope = {
        "customers_df": customers_df, "products_df": products_df,
        "orders_df": orders_df, "order_items_df": order_items_df,
        "returns_df": returns_df, "reviews_df": reviews_df,
        "pd": pd
    }

    try:
        exec(pandas_code, {}, local_scope)
        pandas_answer = local_scope.get("pandas_answer")
        pandas_error = None
    except Exception as e:
        pandas_answer = None
        pandas_error = str(e)

    return {"pandas_code": pandas_code, "pandas_answer": pandas_answer, "pandas_error": pandas_error}

# test on the segment question
verify = dual_verify(
    "What is the total revenue per customer segment, and how many customers are in each segment?",
    outcome2["sql"], outcome2["result"], conn
)
print("Pandas code generated:")
print(verify["pandas_code"])
print()
print("Pandas answer:")
print(verify["pandas_answer"])
print()
print("Pandas error:", verify["pandas_error"])
print()
print("SQL result was:")
print(outcome2["result"])

Pandas code generated:
order_items_merged = order_items_df.merge(
    orders_df[['order_id', 'customer_id']],
    on='order_id',
    how='inner'
)

order_items_merged['line_revenue'] = order_items_merged['quantity'] * order_items_merged['unit_price']

customer_rev = (
    order_items_merged
    .groupby('customer_id')['line_revenue']
    .sum()
    .reset_index()
)

cust_seg_rev = (
    customers_df[['customer_id', 'segment']]
    .merge(customer_rev, on='customer_id', how='left')
)

cust_seg_rev['line_revenue'] = cust_seg_rev['line_revenue'].fillna(0)

pandas_answer = (
    cust_seg_rev
    .groupby('segment')
    .agg(
        total_revenue=('line_revenue', 'sum'),
        customer_count=('customer_id', 'nunique')
    )
    .reset_index()
)

Pandas answer:
     segment  total_revenue  customer_count
0     Retail     3201699.32             103
1        VIP      331059.26              12
2  Wholesale      807845.29              35

Pandas error: None

SQL result was:
     segment  cust

In [21]:
app_code = '''
import streamlit as st
import pandas as pd
import sqlite3
from groq import Groq

st.set_page_config(page_title="AI SQL Analytics Agent", layout="wide")

@st.cache_resource
def load_resources():
    conn = sqlite3.connect("ecommerce.db", check_same_thread=False)
    client = Groq()
    return conn, client

conn, client = load_resources()

schema_description = """
Tables:

customers (customer_id, name, city, signup_date, segment)
products (product_id, name, category, price)
orders (order_id, customer_id, order_date, status)
order_items (order_item_id, order_id, product_id, quantity, unit_price)
returns (return_id, order_item_id, return_date, reason)
reviews (review_id, product_id, customer_id, rating, review_date)

Relationships:
- orders.customer_id -> customers.customer_id
- order_items.order_id -> orders.order_id
- order_items.product_id -> products.product_id
- returns.order_item_id -> order_items.order_item_id
- reviews.product_id -> products.product_id
- reviews.customer_id -> customers.customer_id
"""

def generate_sql(question):
    prompt = f"""You are a SQL analyst. Given this database schema:

{schema_description}

Write a single SQLite query that answers this question: "{question}"

Return ONLY the SQL query, no explanation, no markdown formatting, no backticks.
"""
    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content.strip().replace("```sql", "").replace("```", "").strip()

def self_check(question, sql, result, error):
    review_prompt = f"""You wrote this SQL to answer the question: "{question}"

SQL:
{sql}

{"It failed with this error: " + error if error else "It ran and returned this result: " + result.to_string()}

Review your own query critically. Check specifically for:
- Aggregating at the wrong level of granularity
- Missing GROUP BY causing duplicate counting after a JOIN
- Using INNER JOIN when LEFT JOIN was needed
- Whether the result plausibly answers the question asked

Respond with ONLY one of:
"OK" if correct
"RETRY: <specific reason>" if something is wrong
"""
    return client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": review_prompt}]
    ).choices[0].message.content.strip()

def fix_sql(question, old_sql, reason):
    fix_prompt = f"""Your previous SQL for the question "{question}" had this issue: {reason}

Previous SQL:
{old_sql}

Write a corrected SQLite query. Return ONLY the SQL, no explanation, no markdown.
"""
    return client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": fix_prompt}]
    ).choices[0].message.content.strip().replace("```sql", "").replace("```", "").strip()

def dual_verify(question, conn):
    pandas_prompt = f"""Given these pandas DataFrames loaded from a database:

customers_df, products_df, orders_df, order_items_df, returns_df, reviews_df

Each has the same columns as a SQL table with that name (minus the _df suffix).

Write pandas code (not SQL) that answers this question: "{question}"

The code must end by assigning the final answer to a variable called pandas_answer.
Return ONLY the Python code, no explanation, no markdown, no backticks.
"""
    pandas_code = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": pandas_prompt}]
    ).choices[0].message.content.strip().replace("```python", "").replace("```", "").strip()

    customers_df = pd.read_sql_query("SELECT * FROM customers", conn)
    products_df = pd.read_sql_query("SELECT * FROM products", conn)
    orders_df = pd.read_sql_query("SELECT * FROM orders", conn)
    order_items_df = pd.read_sql_query("SELECT * FROM order_items", conn)
    returns_df = pd.read_sql_query("SELECT * FROM returns", conn)
    reviews_df = pd.read_sql_query("SELECT * FROM reviews", conn)

    local_scope = {
        "customers_df": customers_df, "products_df": products_df,
        "orders_df": orders_df, "order_items_df": order_items_df,
        "returns_df": returns_df, "reviews_df": reviews_df, "pd": pd
    }
    try:
        exec(pandas_code, {}, local_scope)
        return local_scope.get("pandas_answer"), pandas_code, None
    except Exception as e:
        return None, pandas_code, str(e)

def answers_match(sql_result, pandas_answer):
    try:
        if isinstance(pandas_answer, pd.DataFrame):
            sql_sorted = sql_result.reset_index(drop=True)
            pd_sorted = pandas_answer.reset_index(drop=True)
            return sql_sorted.equals(pd_sorted) or (
                sql_sorted.select_dtypes(include="number").round(2).equals(
                    pd_sorted.select_dtypes(include="number").round(2)
                )
            )
        else:
            sql_val = sql_result.iloc[0, 0]
            return abs(float(sql_val) - float(pandas_answer)) < 0.01
    except Exception:
        return None  # comparison itself failed - treat as unverifiable, not a mismatch

st.title("AI SQL Analytics Agent")
st.caption("Ask a business question about the e-commerce database. The agent writes SQL, checks its own work, and verifies the answer independently with pandas.")

question = st.text_input("Your question")

if st.button("Ask") and question:
    with st.spinner("Generating SQL..."):
        sql = generate_sql(question)

    try:
        result = pd.read_sql_query(sql, conn)
        error = None
    except Exception as e:
        result = None
        error = str(e)

    with st.spinner("Reviewing the query..."):
        review = self_check(question, sql, result, error)

    retried = False
    if review.startswith("RETRY"):
        retried = True
        with st.spinner("Fixing the query..."):
            sql = fix_sql(question, sql, review)
        try:
            result = pd.read_sql_query(sql, conn)
            error = None
        except Exception as e:
            result = None
            error = str(e)

    st.markdown("### SQL used")
    if retried:
        st.warning(f"Self-correction triggered: {review}")
    st.code(sql, language="sql")

    if error:
        st.error(f"Query failed: {error}")
    else:
        st.markdown("### Answer")
        st.dataframe(result)

        with st.spinner("Verifying independently with pandas..."):
            pandas_answer, pandas_code, pandas_error = dual_verify(question, conn)

        with st.expander("Independent verification (pandas)"):
            st.code(pandas_code, language="python")
            if pandas_error:
                st.warning(f"Pandas verification could not run: {pandas_error}")
            else:
                match = answers_match(result, pandas_answer)
                if match is True:
                    st.success("Verified: independent pandas computation matches the SQL result.")
                elif match is False:
                    st.error("Mismatch: the independent pandas computation does not match the SQL result. Treat this answer with caution.")
                else:
                    st.info("Could not automatically compare the two results - review both manually.")
                st.write("Pandas result:")
                st.write(pandas_answer)
'''

with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_code)

print("app.py created")

app.py created


In [1]:
app_code = '''
import streamlit as st
import pandas as pd
import sqlite3
from groq import Groq

st.set_page_config(page_title="AI SQL Analytics Agent", layout="wide")

@st.cache_resource
def load_resources():
    conn = sqlite3.connect("ecommerce.db", check_same_thread=False)
    client = Groq()
    return conn, client

conn, client = load_resources()

schema_description = """
Tables:

customers (customer_id, name, city, signup_date, segment)
products (product_id, name, category, price)
orders (order_id, customer_id, order_date, status)
order_items (order_item_id, order_id, product_id, quantity, unit_price)
returns (return_id, order_item_id, return_date, reason)
reviews (review_id, product_id, customer_id, rating, review_date)

Relationships:
- orders.customer_id -> customers.customer_id
- order_items.order_id -> orders.order_id
- order_items.product_id -> products.product_id
- returns.order_item_id -> order_items.order_item_id
- reviews.product_id -> products.product_id
- reviews.customer_id -> customers.customer_id
"""

# Exact same schema text used for BOTH sql and pandas generation, so the two
# paths cannot drift apart on what columns actually exist. This replaces the
# earlier vague description that let the pandas path hallucinate a
# return_quantity column that does not exist in the returns table.
dataframe_schema = """
customers_df columns: customer_id, name, city, signup_date, segment
products_df columns: product_id, name, category, price
orders_df columns: order_id, customer_id, order_date, status
order_items_df columns: order_item_id, order_id, product_id, quantity, unit_price
returns_df columns: return_id, order_item_id, return_date, reason
reviews_df columns: review_id, product_id, customer_id, rating, review_date

IMPORTANT: the returns_df table records THAT an order_item was returned
(one row per return event). It does NOT have a return_quantity column or
any partial-quantity tracking. To count returns, count rows or use
order_item_id membership - do not assume a quantity field exists on
returns_df.
"""

def generate_sql(question):
    prompt = f"""You are a SQL analyst. Given this database schema:

{schema_description}

Write a single SQLite query that answers this question: "{question}"

Return ONLY the SQL query, no explanation, no markdown formatting, no backticks.
"""
    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content.strip().replace("```sql", "").replace("```", "").strip()

def self_check(question, sql, result, error):
    review_prompt = f"""You wrote this SQL to answer the question: "{question}"

SQL:
{sql}

{"It failed with this error: " + error if error else "It ran and returned this result: " + result.to_string()}

Review your own query critically. Check specifically for:
- Aggregating at the wrong level of granularity
- Missing GROUP BY causing duplicate counting after a JOIN
- Using INNER JOIN when LEFT JOIN was needed
- Whether the result plausibly answers the question asked

Respond with ONLY one of:
"OK" if correct
"RETRY: <specific reason>" if something is wrong
"""
    return client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": review_prompt}]
    ).choices[0].message.content.strip()

def fix_sql(question, old_sql, reason):
    fix_prompt = f"""Your previous SQL for the question "{question}" had this issue: {reason}

Previous SQL:
{old_sql}

Write a corrected SQLite query. Return ONLY the SQL, no explanation, no markdown.
"""
    return client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": fix_prompt}]
    ).choices[0].message.content.strip().replace("```sql", "").replace("```", "").strip()

def dual_verify(question, conn):
    pandas_prompt = f"""Given these pandas DataFrames, already loaded, with EXACTLY these columns:

{dataframe_schema}

Write pandas code (not SQL) that answers this question: "{question}"

Only use columns listed above. Do not assume a column exists if it is not
listed. The code must end by assigning the final answer to a variable
called pandas_answer.
Return ONLY the Python code, no explanation, no markdown, no backticks.
"""
    pandas_code = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": pandas_prompt}]
    ).choices[0].message.content.strip().replace("```python", "").replace("```", "").strip()

    customers_df = pd.read_sql_query("SELECT * FROM customers", conn)
    products_df = pd.read_sql_query("SELECT * FROM products", conn)
    orders_df = pd.read_sql_query("SELECT * FROM orders", conn)
    order_items_df = pd.read_sql_query("SELECT * FROM order_items", conn)
    returns_df = pd.read_sql_query("SELECT * FROM returns", conn)
    reviews_df = pd.read_sql_query("SELECT * FROM reviews", conn)

    local_scope = {
        "customers_df": customers_df, "products_df": products_df,
        "orders_df": orders_df, "order_items_df": order_items_df,
        "returns_df": returns_df, "reviews_df": reviews_df, "pd": pd
    }
    try:
        exec(pandas_code, {}, local_scope)
        return local_scope.get("pandas_answer"), pandas_code, None
    except Exception as e:
        return None, pandas_code, str(e)

def answers_match(sql_result, pandas_answer):
    try:
        if isinstance(pandas_answer, pd.DataFrame):
            sql_sorted = sql_result.reset_index(drop=True)
            pd_sorted = pandas_answer.reset_index(drop=True)
            return sql_sorted.equals(pd_sorted) or (
                sql_sorted.select_dtypes(include="number").round(2).equals(
                    pd_sorted.select_dtypes(include="number").round(2)
                )
            )
        else:
            sql_val = sql_result.iloc[0, 0]
            return abs(float(sql_val) - float(pandas_answer)) < 0.01
    except Exception:
        return None

st.title("AI SQL Analytics Agent")
st.caption("Ask a business question about the e-commerce database. The agent writes SQL, checks its own work, and verifies the answer independently with pandas.")

question = st.text_input("Your question")

if st.button("Ask") and question:
    with st.spinner("Generating SQL..."):
        sql = generate_sql(question)

    try:
        result = pd.read_sql_query(sql, conn)
        error = None
    except Exception as e:
        result = None
        error = str(e)

    with st.spinner("Reviewing the query..."):
        review = self_check(question, sql, result, error)

    retried = False
    if review.startswith("RETRY"):
        retried = True
        with st.spinner("Fixing the query..."):
            sql = fix_sql(question, sql, review)
        try:
            result = pd.read_sql_query(sql, conn)
            error = None
        except Exception as e:
            result = None
            error = str(e)

    st.markdown("### SQL used")
    if retried:
        st.warning(f"Self-correction triggered: {review}")
    st.code(sql, language="sql")

    if error:
        st.error(f"Query failed: {error}")
    else:
        st.markdown("### Answer")
        st.dataframe(result)

        with st.spinner("Verifying independently with pandas..."):
            pandas_answer, pandas_code, pandas_error = dual_verify(question, conn)

        with st.expander("Independent verification (pandas)", expanded=True):
            st.code(pandas_code, language="python")
            if pandas_error:
                st.warning(f"Pandas verification could not run: {pandas_error}")
            else:
                match = answers_match(result, pandas_answer)
                if match is True:
                    st.success("Verified: independent pandas computation matches the SQL result.")
                elif match is False:
                    st.error("Mismatch: the independent pandas computation does not match the SQL result. Treat this answer with caution.")
                else:
                    st.info("Could not automatically compare the two results - review both manually.")
                st.write("Pandas result:")
                st.write(pandas_answer)
'''

with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_code)

print("app.py updated")

app.py updated


In [2]:
app_code = '''
import streamlit as st
import pandas as pd
import sqlite3
from groq import Groq

st.set_page_config(page_title="AI SQL Analytics Agent", layout="wide")

@st.cache_resource
def load_resources():
    conn = sqlite3.connect("ecommerce.db", check_same_thread=False)
    client = Groq()
    return conn, client

conn, client = load_resources()

schema_description = """
Tables:

customers (customer_id, name, city, signup_date, segment)
products (product_id, name, category, price)
orders (order_id, customer_id, order_date, status)
order_items (order_item_id, order_id, product_id, quantity, unit_price)
returns (return_id, order_item_id, return_date, reason)
reviews (review_id, product_id, customer_id, rating, review_date)

Relationships:
- orders.customer_id -> customers.customer_id
- order_items.order_id -> orders.order_id
- order_items.product_id -> products.product_id
- returns.order_item_id -> order_items.order_item_id
- reviews.product_id -> products.product_id
- reviews.customer_id -> customers.customer_id

IMPORTANT DEFINITION: "revenue" and "total revenue" mean NET revenue -
i.e. revenue from order_items EXCLUDING any order_item that appears in the
returns table (a returned item generated no final revenue). Always exclude
returned order_items (via a LEFT JOIN ... IS NULL, or NOT IN a subquery
against returns.order_item_id) when computing any revenue figure, unless
the question explicitly asks for gross/pre-return revenue.
"""

dataframe_schema = """
customers_df columns: customer_id, name, city, signup_date, segment
products_df columns: product_id, name, category, price
orders_df columns: order_id, customer_id, order_date, status
order_items_df columns: order_item_id, order_id, product_id, quantity, unit_price
returns_df columns: return_id, order_item_id, return_date, reason
reviews_df columns: review_id, product_id, customer_id, rating, review_date

IMPORTANT: the returns_df table records THAT an order_item was returned
(one row per return event). It does NOT have a return_quantity column or
any partial-quantity tracking. To count returns, count rows or use
order_item_id membership - do not assume a quantity field exists on
returns_df.

IMPORTANT DEFINITION: "revenue" and "total revenue" mean NET revenue -
exclude any order_item whose order_item_id appears in returns_df, unless
the question explicitly asks for gross/pre-return revenue.
"""

def generate_sql(question):
    prompt = f"""You are a SQL analyst. Given this database schema:

{schema_description}

Write a single SQLite query that answers this question: "{question}"

Return ONLY the SQL query, no explanation, no markdown formatting, no backticks.
"""
    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content.strip().replace("```sql", "").replace("```", "").strip()

def self_check(question, sql, result, error):
    review_prompt = f"""You wrote this SQL to answer the question: "{question}"

SQL:
{sql}

{"It failed with this error: " + error if error else "It ran and returned this result: " + result.to_string()}

Review your own query critically. Check specifically for:
- Aggregating at the wrong level of granularity
- Missing GROUP BY causing duplicate counting after a JOIN
- Using INNER JOIN when LEFT JOIN was needed
- Whether revenue figures correctly exclude returned order_items
- Whether the result plausibly answers the question asked

Respond with ONLY one of:
"OK" if correct
"RETRY: <specific reason>" if something is wrong
"""
    return client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": review_prompt}]
    ).choices[0].message.content.strip()

def fix_sql(question, old_sql, reason):
    fix_prompt = f"""Your previous SQL for the question "{question}" had this issue: {reason}

Previous SQL:
{old_sql}

Write a corrected SQLite query. Return ONLY the SQL, no explanation, no markdown.
"""
    return client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": fix_prompt}]
    ).choices[0].message.content.strip().replace("```sql", "").replace("```", "").strip()

def dual_verify(question, conn):
    pandas_prompt = f"""Given these pandas DataFrames, already loaded, with EXACTLY these columns:

{dataframe_schema}

Write pandas code (not SQL) that answers this question: "{question}"

Only use columns listed above. Do not assume a column exists if it is not
listed. The code must end by assigning the final answer to a variable
called pandas_answer.
Return ONLY the Python code, no explanation, no markdown, no backticks.
"""
    pandas_code = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": pandas_prompt}]
    ).choices[0].message.content.strip().replace("```python", "").replace("```", "").strip()

    customers_df = pd.read_sql_query("SELECT * FROM customers", conn)
    products_df = pd.read_sql_query("SELECT * FROM products", conn)
    orders_df = pd.read_sql_query("SELECT * FROM orders", conn)
    order_items_df = pd.read_sql_query("SELECT * FROM order_items", conn)
    returns_df = pd.read_sql_query("SELECT * FROM returns", conn)
    reviews_df = pd.read_sql_query("SELECT * FROM reviews", conn)

    local_scope = {
        "customers_df": customers_df, "products_df": products_df,
        "orders_df": orders_df, "order_items_df": order_items_df,
        "returns_df": returns_df, "reviews_df": reviews_df, "pd": pd
    }
    try:
        exec(pandas_code, {}, local_scope)
        return local_scope.get("pandas_answer"), pandas_code, None
    except Exception as e:
        return None, pandas_code, str(e)

def answers_match(sql_result, pandas_answer):
    """
    Returns True (matches), False (mismatch), or None (could not compare).
    Handles: single-value results, DataFrame results, and dict results
    (a common shape the pandas path produces for multi-field answers).
    """
    try:
        if isinstance(pandas_answer, dict):
            if sql_result.shape[0] != 1:
                return None
            row = sql_result.iloc[0]
            numeric_keys = [k for k, v in pandas_answer.items() if isinstance(v, (int, float))]
            if not numeric_keys:
                return None
            mismatches = []
            for k in numeric_keys:
                matches_a_column = any(
                    k.lower() in str(col).lower() or str(col).lower() in k.lower()
                    for col in sql_result.columns
                )
                if not matches_a_column:
                    continue
                best_col = min(
                    sql_result.columns,
                    key=lambda c: abs(len(str(c)) - len(k))
                )
                try:
                    sql_val = float(row[best_col])
                    pd_val = float(pandas_answer[k])
                    if abs(sql_val - pd_val) > max(0.01, abs(sql_val) * 0.001):
                        mismatches.append((k, sql_val, pd_val))
                except Exception:
                    continue
            if not mismatches:
                return None if not numeric_keys else True
            return False

        elif isinstance(pandas_answer, pd.DataFrame):
            sql_sorted = sql_result.reset_index(drop=True)
            pd_sorted = pandas_answer.reset_index(drop=True)
            return sql_sorted.equals(pd_sorted) or (
                sql_sorted.select_dtypes(include="number").round(2).equals(
                    pd_sorted.select_dtypes(include="number").round(2)
                )
            )
        else:
            sql_val = sql_result.iloc[0, 0]
            return abs(float(sql_val) - float(pandas_answer)) < 0.01
    except Exception:
        return None

st.title("AI SQL Analytics Agent")
st.caption("Ask a business question about the e-commerce database. The agent writes SQL, checks its own work, and verifies the answer independently with pandas.")

question = st.text_input("Your question")

if st.button("Ask") and question:
    with st.spinner("Generating SQL..."):
        sql = generate_sql(question)

    try:
        result = pd.read_sql_query(sql, conn)
        error = None
    except Exception as e:
        result = None
        error = str(e)

    with st.spinner("Reviewing the query..."):
        review = self_check(question, sql, result, error)

    retried = False
    if review.startswith("RETRY"):
        retried = True
        with st.spinner("Fixing the query..."):
            sql = fix_sql(question, sql, review)
        try:
            result = pd.read_sql_query(sql, conn)
            error = None
        except Exception as e:
            result = None
            error = str(e)

    st.markdown("### SQL used")
    if retried:
        st.warning(f"Self-correction triggered: {review}")
    st.code(sql, language="sql")

    if error:
        st.error(f"Query failed: {error}")
    else:
        st.markdown("### Answer")
        st.dataframe(result)

        with st.spinner("Verifying independently with pandas..."):
            pandas_answer, pandas_code, pandas_error = dual_verify(question, conn)

        with st.expander("Independent verification (pandas)", expanded=True):
            st.code(pandas_code, language="python")
            if pandas_error:
                st.warning(f"Pandas verification could not run: {pandas_error}")
            else:
                match = answers_match(result, pandas_answer)
                if match is True:
                    st.success("Verified: independent pandas computation matches the SQL result.")
                elif match is False:
                    st.error("Mismatch: the independent pandas computation does not match the SQL result. Treat this answer with caution.")
                else:
                    st.info("Could not automatically compare the two results - review both manually.")
                st.write("Pandas result:")
                st.write(pandas_answer)
'''

with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_code)

print("app.py updated")

app.py updated


In [3]:
def _normalize(s):
    return set(str(s).lower().replace("_", " ").split())

In [1]:
def _normalize(s):
    return set(str(s).lower().replace("_", " ").split())

def answers_match(sql_result, pandas_answer):
    try:
        if isinstance(pandas_answer, dict):
            if sql_result.shape[0] != 1:
                return None
            row = sql_result.iloc[0]
            numeric_keys = [k for k, v in pandas_answer.items() if isinstance(v, (int, float))]
            if not numeric_keys:
                return None
            checked = 0
            mismatches = []
            for k in numeric_keys:
                k_tokens = _normalize(k)
                best_col = None
                best_overlap = 0
                for col in sql_result.columns:
                    overlap = len(k_tokens & _normalize(col))
                    if overlap > best_overlap:
                        best_overlap = overlap
                        best_col = col
                if best_col is None or best_overlap == 0:
                    continue
                try:
                    sql_val = float(row[best_col])
                    pd_val = float(pandas_answer[k])
                    checked += 1
                    if abs(sql_val - pd_val) > max(0.01, abs(sql_val) * 0.001):
                        mismatches.append((k, best_col, sql_val, pd_val))
                except Exception:
                    continue
            if checked == 0:
                return None
            return len(mismatches) == 0

        elif isinstance(pandas_answer, pd.DataFrame):
            sql_sorted = sql_result.reset_index(drop=True)
            pd_sorted = pandas_answer.reset_index(drop=True)
            return sql_sorted.equals(pd_sorted) or (
                sql_sorted.select_dtypes(include="number").round(2).equals(
                    pd_sorted.select_dtypes(include="number").round(2)
                )
            )
        else:
            sql_val = sql_result.iloc[0, 0]
            return abs(float(sql_val) - float(pandas_answer)) < 0.01
    except Exception:
        return None

In [2]:


import os, shutil

submission_dir = r"C:\Users\NAAJI\Naji_SQL_Analytics_Agent"
os.makedirs(os.path.join(submission_dir, "Notebook"), exist_ok=True)
os.makedirs(os.path.join(submission_dir, "Screenshots"), exist_ok=True)

shutil.copy("app.py", submission_dir)
shutil.copy("ecommerce.db", submission_dir)

print("Created:", submission_dir)
print("Still need: your .ipynb into Notebook/, and screenshots into Screenshots/")








Created: C:\Users\NAAJI\Naji_SQL_Analytics_Agent
Still need: your .ipynb into Notebook/, and screenshots into Screenshots/
